purpose of this notebook is to evaluate on the static embedding model `minishlab/potion-retrieval-32M`

In [1]:
from model2vec import StaticModel
from sentence_transformers import SentenceTransformer
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [2]:
from datasets import load_dataset, DatasetDict

def load_data(repo="frankjc2022/semantic-history-search"):

    docs = load_dataset(repo, name="docs")["train"]
    queries = load_dataset(repo, name="queries")["train"]
    qrels = load_dataset(repo, name="qrels")["train"]
    
    return DatasetDict(docs=docs, queries=queries, qrels=qrels)

ds = load_data()
docs = ds["docs"].to_pandas()
queries = ds["queries"].to_pandas()
qrels = ds["qrels"].to_pandas()
query_pairs = queries.merge(qrels[["query_id", "doc_id", "relevance", "rank"]], on="query_id").merge(docs[["doc_id", "url"]], on="doc_id").drop_duplicates()

# query_pairs.loc[query_pairs["is_multi"]==True]

In [3]:
persona_name = "025_creative_hobbyist"
# persona_name = "chidam"

# MODEL_NAME = "Xenova/all-MiniLM-L6-v2" 
MODEL_NAME = "minishlab/potion-retrieval-32M"

GOLDEN_QUERY_COLS = ["search_query","url"]
PROFILE_COLS = ["url", "title", "description", "frecency", "last_visit_date"] #"url_hash"

# PROFILE_FILENAMES = {
#     "creative_hobyist": {
#         "golden_query_data" : "../test_data/creative_hobyist_profile_golden_queries.csv",
#         "profile_data": "../test_data/creative_hobbyist.csv"
#     },
#     # "chidam": {
#     #     "golden_query_data" : "../data/chidam_golden_query.csv",
#     #     "profile_data": "../data/history_output_file.csv"
#     # }
# }
# PROFILE_FILENAME = "../data/chidam_golden_query.csv"


In [4]:
def get_model(MODEL_NAME):
    if MODEL_NAME == "Xenova/all-MiniLM-L6-v2":
        return SentenceTransformer("all-MiniLM-L6-v2")
    elif MODEL_NAME == "minishlab/potion-retrieval-32M":
        return StaticModel.from_pretrained("minishlab/potion-retrieval-32M")

def get_profile_filenames(persona_name):
    # if persona_name not in PROFILE_FILENAMES.keys():
    #     return ValueError(f"persona_name entered not in {list(PROFILE_FILENAMES.keys())}")
    # golden_queries_df = pd.read_csv(PROFILE_FILENAMES[persona_name]['golden_query_data'])[GOLDEN_QUERY_COLS]
    # profile_df = pd.read_csv(PROFILE_FILENAMES[persona_name]["profile_data"])
    #[PROFILE_COLS]

    profile_df = docs.loc[docs["profile"]==persona_name]
    golden_queries_df = query_pairs.loc[query_pairs["profile"]==persona_name]
    
    return golden_queries_df, profile_df

In [5]:
model = get_model(MODEL_NAME)

In [6]:
golden_queries_df, profile_df = get_profile_filenames(persona_name)
golden_queries_df

,query_id,search_query,profile,profile_id,is_temporal,is_multi,variant,ref_datetime_iso,doc_id,relevance,rank,url
2500,177fd25d37deb5f7,Best mosquito repellent outdoor,025_creative_hobbyist,944346ce7b090355,False,False,base,,8eea7bb8edd43f0f,1,1,http://www.wikihow.com/Get-Rid-of-Mosquitoes-i...
2501,5f3ca5774311ff47,How to get rid of heat stains on wood,025_creative_hobbyist,944346ce7b090355,False,False,base,,c9c78eebd35d316e,1,1,http://tipnut.com/diy-how-to-remove-white-heat...
2502,4401edf823ba603a,Zucchini recipes,025_creative_hobbyist,944346ce7b090355,False,False,base,,9d03a87c724f246f,1,1,http://startcooking.com/how-to-zucchini
2503,88be8c028c7cd5ee,Planting grass,025_creative_hobbyist,944346ce7b090355,False,False,base,,d7d50a3d458ce208,1,1,http://www.wikihow.com/Grow-Pampas-Grass
2504,1050d6bc10ac7ae3,Hamstring remedies,025_creative_hobbyist,944346ce7b090355,False,False,base,,bd3454d78d774133,1,1,http://www.runnersworld.com/for-beginners-only...
2505,4d30a5472ad4cb96,Organic fertilizers,025_creative_hobbyist,944346ce7b090355,False,False,base,,609ec7a855d96e0a,1,1,http://www.sunset.com/garden/garden-basics/cra...
2506,ee7ed0b2b3aebf92,Whats a choir,025_creative_hobbyist,944346ce7b090355,False,False,base,,c58e818b0211a06b,1,1,https://simple.wikipedia.org/wiki/Choir_(music)
2507,9da5f91e391800d8,Make money from music,025_creative_hobbyist,944346ce7b090355,False,False,base,,07e5bb5a4fce58f5,1,1,http://hiphopmakers.com/how-to-sell-beats-onli...
2508,e250e7d407621cd6,Chlorine for pool,025_creative_hobbyist,944346ce7b090355,False,False,base,,ec5d7c6332babf04,1,1,https://www.inyopools.com/blog/what-type-of-po...
2509,dd4e5c060b177dcd,Patio decoration,025_creative_hobbyist,944346ce7b090355,False,False,base,,4964380a087e8262,1,1,https://www.landscapingnetwork.com/patios/conc...


In [7]:
profile_df

,doc_id,url,title,description,frecency,last_visit_date,profile,profile_id,variant
82653,3b9332b3e77dcbaf,http://www.ehow.com/how_5409995_cook-fennel.html,How to Cook Fennel,"A decidedly odd-looking vegetable, fennel rese...",5000,1738272723614003,025_creative_hobbyist,944346ce7b090355,base
82654,c2a01b4478300d0f,http://www.phschool.com/language_arts/,Language Arts,Language Arts Student Resources Textbook Compa...,5000,1738272733950727,025_creative_hobbyist,944346ce7b090355,base
82655,5b4098fca631d4d5,https://www.8notes.com/biographies/adams.asp,Items to buy by Bryan Adams,Items to buy by Bryan Adams (Everything I Do) ...,4999,1738272724722098,025_creative_hobbyist,944346ce7b090355,base
82656,8eea7bb8edd43f0f,http://www.wikihow.com/Get-Rid-of-Mosquitoes-i...,How to Get Rid of Mosquitoes in Your Yard,1 Drain any areas with standing water. Mosquit...,4998,1738272735386823,025_creative_hobbyist,944346ce7b090355,base
82657,fa865024f44bc6a4,http://www.goddessgift.com/goddess-myths/godde...,Goddess Symbols: Hestia,Goddess Symbols: Hestia Goddess Symbols and Sa...,4997,1738272735282816,025_creative_hobbyist,944346ce7b090355,base
...,...,...,...,...,...,...,...,...,...
88900,c62fb10d4c75e929,http://www.cnn.com/2008/TRAVEL/traveltips/06/2...,How to get airport lounge discounts,By Andrea Bennett (Travel + Leisure) -- The fo...,107,1738272729266412,025_creative_hobbyist,944346ce7b090355,base
88901,fb630ebaf0bee9b7,https://www.diabetesselfmanagement.com/managin...,What Makes Blood Glucose Go Up or Down?,What Makes Blood Glucose Go Up or Down? Update...,106,1738272724750099,025_creative_hobbyist,944346ce7b090355,base
88902,fcac6fa5e6fa11e0,https://fit4less.ca/facts,Fit4Less Facts,Fit4Less Facts Membership Card You must have y...,105,1738272735062801,025_creative_hobbyist,944346ce7b090355,base
88903,4a80c9217a88c02d,https://blog.onestoppoppyshoppe.com/articles/p...,One Stop Poppy Shoppe Blog,Poppy Flower Seeds – Germinating and Growing P...,103,1738272734902791,025_creative_hobbyist,944346ce7b090355,base


In [8]:
%%time
profile_embeddings = model.encode(profile_df['title'].str.lower().values.tolist())

CPU times: user 162 ms, sys: 22.9 ms, total: 185 ms
Wall time: 65.7 ms


In [9]:


def search(query, model, profile_embeddings, profile_df, top_k=2, threshold=0.6):
    query_embeddings = model.encode([query])
    sim_scores = cosine_similarity(query_embeddings, profile_embeddings)[0]  # shape: (N,)

    valid_idx = np.where(sim_scores >= threshold)[0]
    if valid_idx.size == 0:
        return []

    ranked_idx = valid_idx[np.argsort(-sim_scores[valid_idx])[:top_k]]
    return profile_df.iloc[ranked_idx]['url'].tolist()


In [10]:
search("Best mosquito repellent outdoor", model, profile_embeddings, profile_df, top_k=5)
search("Zucchini recipes", model, profile_embeddings, profile_df, top_k=2)

['http://startcooking.com/how-to-zucchini',
 'http://allrecipes.com/recipe/233453/low-carb-zucchini-pasta/']

In [11]:
correct = 0
for idx, row in golden_queries_df.iterrows():
    actual = row['url']
    preds = search(row['search_query'].lower(), model, profile_embeddings, profile_df, top_k=3, threshold=0.5)
    if actual in preds:
        correct += 1
        
    # else:
    #     print(idx, row['search_query'], actual, preds)
        


In [12]:
correct

30

In [13]:
correct/len(golden_queries_df)

0.8108108108108109

In [14]:
## Results for creative_hobyist (38 queries)
## allXenova/all-MiniLM-L6-v2: 33 correct (0.84)
## minishlab/potion-retrieval-32M: 30 correct (0.78)

In [15]:
## Results for chidam (48 queries)
## allXenova/all-MiniLM-L6-v2: 17 correct (0.35)
## minishlab/potion-retrieval-32M: 20 correct (0.40)

#### Using long & short Q datasets and profiles

In [16]:
# longQ = [
#       "alternative medicine to manage ringing in my ear",
#       "solutions for noisy neighbors",
#       "how to train a cat to use the litter box",
#       "how to make your dog loss weight",
#       "what are the best foods for dogs",
#       "how to keep your kids safe online",
#       "some recommendations for vegan recipes for kids",
#       "best places to visit in new york in summer",
#       "how much does it cost to go to niagara falls",
#       "how to take screenshot on windows 11",
#       "what are some of the symptoms of high blood pressure",
#       "what are some treatment options for stroke",
#       "organic dog food for sensitive stomachs",
#       "best budget-friendly family vacation spots in California",
#       "what are some popular open source projects",
#     ]

# shortQ = [
#       "healthy rice",
#       "endodontist",
#       "arsenal",
#       "gavaskar border trophy",
#       "grand slam champion",
#       "capital gain",
#       "credit limit",
#       "annuity",
#       "physiology",
#       "myopia",
#       "amyloidosis",
#       "alopecia",
#       "dermatitis",
#       "coffee shop",
#       "ball drop",
#     ]

queries_longQ_longD = queries.loc[queries["profile"]=="longQ_longD"]
queries_shortQ_longD = queries.loc[queries["profile"]=="shortQ_longD"]

queries_longQ_shortD = queries.loc[queries["profile"]=="longQ_shortD"]
queries_shortQ_shortD = queries.loc[queries["profile"]=="shortQ_shortD"]

In [17]:
# len(longQ), len(shortQ)
len(queries_longQ_longD), len(queries_shortQ_longD), len(queries_longQ_shortD), len(queries_shortQ_shortD)

(15, 15, 15, 15)

In [18]:
# lonQ_longD_file_path = "../test_data/longQ_longD.json"
# lonQ_shortD_file_path = "../test_data/longQ_shortD.json"

# shortQ_longD_file_path = "../test_data/shortQ_longD.json"
# shortQ_shortD_file_path = "../test_data/shortQ_shortD.json"

In [19]:
# lonQ_longD_df = pd.read_json(lonQ_longD_file_path)[PROFILE_COLS]
# lonQ_shortD_df = pd.read_json(lonQ_shortD_file_path)[PROFILE_COLS]

# shortQ_longD_df = pd.read_json(shortQ_longD_file_path)[PROFILE_COLS]
# shortQ_shortD_df = pd.read_json(shortQ_shortD_file_path)[PROFILE_COLS]

lonQ_longD_df = docs.loc[docs["profile"] == "longQ_longD"]
lonQ_shortD_df = docs.loc[docs["profile"] == "longQ_shortD"]
shortQ_longD_df = docs.loc[docs["profile"] == "shortQ_longD"]
shortQ_shortD_df = docs.loc[docs["profile"] == "shortQ_shortD"]


lonQ_longD_df.shape, lonQ_shortD_df.shape, shortQ_longD_df.shape, shortQ_shortD_df.shape

((8743, 9), (7411, 9), (8717, 9), (7444, 9))

In [20]:
lonQ_longD_df[:36]

,doc_id,url,title,description,frecency,last_visit_date,profile,profile_id,variant
88905,8bf825df90d583f7,https://www.buoyhealth.com/symptoms-a-z/ringin...,"Ringing in the Ears Symptoms, Causes & Treatme...","Ringing in the Ears Symptoms, Causes & Treatme...",1421560,1736196807185413,longQ_longD,9c27e185d18e183a,length-variants
88906,a09ef53a5b37d6c4,https://www.necksolutions.com/tinnitus.html,Tinnitus Is Often Described As Ringing In The ...,Tinnitus Is Often Described As Ringing In The ...,281700,1736189753947959,longQ_longD,9c27e185d18e183a,length-variants
88907,1261bcedfb073438,https://www.legalzoom.com/articles/neighbor-di...,Neighbor Disputes: What to Do When Your Neighb...,Neighbor Disputes: What to Do When Your Neighb...,274017,1736176190746123,longQ_longD,9c27e185d18e183a,length-variants
88908,082a5ff5b004238b,https://answers.yahoo.com/question/index?qid=2...,"My Microwave is making a loud humming noise, c...",Home & Garden Maintenance & Repairs My Microwa...,248551,1736176755662276,longQ_longD,9c27e185d18e183a,length-variants
88909,7dbf634bacf2cefb,http://www.drsfostersmith.com/pic/article.cfm?...,Frequently Asked Questions about Feline Litter...,Frequently Asked Questions about Feline Litter...,236911,1735422684696590,longQ_longD,9c27e185d18e183a,length-variants
88910,633bccf2f6ecc1f6,https://www.drelseys.com/2015/05/08/how-often-...,Our Blog ::: How Often Should I Change the Lit...,"May 8, 2015How Often Should I Change the Litte...",203868,1734619916609214,longQ_longD,9c27e185d18e183a,length-variants
88911,919a39037c9b416c,https://answers.yahoo.com/question/index?qid=2...,How do you show your dog your a pack leader?,Pets Dogs How do you show your dog your a pack...,157336,1734757831526359,longQ_longD,9c27e185d18e183a,length-variants
88912,6bd7e1ab5ad77fba,http://www.vetwest.com.au/pet-library/weight-t...,Weight - the ideal bodyweight range for your d...,Weight - the ideal bodyweight range for your d...,84530,1735835977907487,longQ_longD,9c27e185d18e183a,length-variants
88913,a0de36e9962a78f0,http://www.dogfoodscoop.com/10-best-dog-food.html,Top 10 Best Dog Food Our Pick of the Best Dry ...,Top 10 Best Dog Food Our Pick of the Best Dry ...,83355,1735829441460524,longQ_longD,9c27e185d18e183a,length-variants
88914,3668678172562360,http://animals.howstuffworks.com/pets/10-fruit...,10 Fruits and Veggies that Aid in Dogs' Nutrition,NEXTStart the Countdown Those farmstand vegeta...,80600,1736177187460005,longQ_longD,9c27e185d18e183a,length-variants


In [21]:
def search_long_short_queries(query_id, model, profile_embeddings, profile_df, top_k=2, threshold=0.6):
    query_details = queries.loc[queries["query_id"]==query_id]
    query = query_details["search_query"].item()

    q_docs = query_pairs.loc[query_pairs["query_id"]==query_id][["doc_id", "rank", "relevance"]].sort_values("rank", ascending=True)
    
    query_embeddings = model.encode([query])
    sim_scores = cosine_similarity(query_embeddings, profile_embeddings)[0]

    valid_idx = np.where(sim_scores >= threshold)[0]
    ranked_idx = valid_idx[np.argsort(-sim_scores[valid_idx])[:top_k]]
    
    if valid_idx.size == 0:
        return []
    
    res = []
    for idx in ranked_idx:
        res.append(profile_df.iloc[idx])

    doc_ids = set(q_docs["doc_id"].astype(str))

    has_overlap = any(str(p.doc_id) in doc_ids for p in res)
    return has_overlap
    
    # ranked_idx = valid_idx[np.argsort(-sim_scores[valid_idx])[:top_k]]
    # return profile_df.iloc[ranked_idx]['url'].tolist()


In [22]:
lonQ_longD_profile_embeddings = model.encode(lonQ_longD_df['title'].str.lower() + " " + lonQ_longD_df['description'].str.lower())
print(lonQ_longD_profile_embeddings.shape)
correct_longQ_longD = 0
for qid in queries_longQ_longD["query_id"]:
    res = search_long_short_queries(qid, model, lonQ_longD_profile_embeddings, lonQ_longD_df, top_k=2, threshold=0.6)
    if res:
        correct_longQ_longD += 1

print("correct_longQ_longD", correct_longQ_longD)
print("correct percentage", round(correct_longQ_longD/len(queries_longQ_longD), 3))

(8743, 512)
correct_longQ_longD 10
correct percentage 0.667


In [23]:
lonQ_shortD_profile_embeddings = model.encode(lonQ_shortD_df['title'].str.lower() + " " + lonQ_shortD_df['description'].str.lower())
print(lonQ_shortD_profile_embeddings.shape)
correct_longQ_shortD = 0
for qid in queries_longQ_shortD["query_id"]:
    res = search_long_short_queries(qid, model, lonQ_shortD_profile_embeddings, lonQ_shortD_df, top_k=2, threshold=0.6)
    if res:
        correct_longQ_shortD += 1

print("correct_longQ_shortD", correct_longQ_shortD)
print("correct percentage", round(correct_longQ_shortD/len(queries_longQ_shortD), 3))

(7411, 512)
correct_longQ_shortD 14
correct percentage 0.933


In [24]:
shortQ_longD_profile_embeddings = model.encode(shortQ_longD_df['title'].str.lower() + " " + shortQ_longD_df['description'].str.lower())
print(shortQ_longD_profile_embeddings.shape)
correct_shortQ_longD = 0
for qid in queries_shortQ_longD["query_id"]:
    res = search_long_short_queries(qid, model, shortQ_longD_profile_embeddings, shortQ_longD_df, top_k=2, threshold=0.6)
    if res:
        correct_shortQ_longD += 1

print("correct_shortQ_longD", correct_shortQ_longD)
print("correct percentage", round(correct_shortQ_longD/len(queries_shortQ_longD), 3))

(8717, 512)
correct_shortQ_longD 10
correct percentage 0.667


In [25]:
shortQ_shortD_profile_embeddings = model.encode(shortQ_shortD_df['title'].str.lower() + " " + shortQ_shortD_df['description'].str.lower())
print(shortQ_shortD_profile_embeddings.shape)
correct_shortQ_shortD = 0
for qid in queries_shortQ_shortD["query_id"]:
    res = search_long_short_queries(qid, model, shortQ_shortD_profile_embeddings, shortQ_shortD_df, top_k=2, threshold=0.6)
    if res:
        correct_shortQ_shortD += 1

print("correct_shortQ_longD", correct_shortQ_shortD)
print("correct percentage", round(correct_shortQ_shortD/len(queries_shortQ_shortD), 3))

(7444, 512)
correct_shortQ_longD 14
correct percentage 0.933


In [26]:
## store these in sqlite-vec DB

In [27]:
import numpy as np
import sqlite3
import sqlite_vec

from typing import List
import struct

In [28]:
def serialize_f32(vector: List[float]) -> bytes:
    """serializes a list of floats into a compact "raw bytes" format"""
    return struct.pack("%sf" % len(vector), *vector)

In [29]:
db = sqlite3.connect(":memory:")
db.enable_load_extension(True)
sqlite_vec.load(db)
db.enable_load_extension(False)

sqlite_version, vec_version = db.execute(
    "select sqlite_version(), vec_version()"
).fetchone()
print(f"sqlite_version={sqlite_version}, vec_version={vec_version}")


sqlite_version=3.50.2, vec_version=v0.1.6


In [30]:
lonQ_shortD_profile_embeddings.shape

(7411, 512)

In [31]:
persona = "lonQ_shortD"
EMBEDDING_SIZE = lonQ_shortD_profile_embeddings.shape[1]
db.execute(f"CREATE VIRTUAL TABLE vec_items_{persona}_2 USING vec0(embedding float[{EMBEDDING_SIZE}], embedding_coarse bit[{EMBEDDING_SIZE}])")

with db:
    for idx, vec in enumerate(lonQ_shortD_profile_embeddings):
        embedding = serialize_f32(vec)
        db.execute(
            f"INSERT INTO vec_items_{persona}_2(rowid, embedding, embedding_coarse) VALUES (?, ?, vec_quantize_binary(?))",
            [idx, embedding, embedding],  # Convert vector to binary format
        )


In [32]:
# longQ
longQ = queries_longQ_shortD["search_query"].to_list()
longQ

['alternative medicine to manage ringing in my ear',
 'solutions for noisy neighbors',
 'how to train a cat to use the litter box',
 'how to make your dog loss weight',
 'what are the best foods for dogs',
 'how to keep your kids safe online',
 'some recommendations for vegan recipes for kids',
 'best places to visit in new york in summer',
 'how much does it cost to go to niagara falls',
 'how to take screenshot on windows 11',
 'what are some of the symptoms of high blood pressure',
 'what are some treatment options for stroke',
 'organic dog food for sensitive stomachs',
 'best budget-friendly family vacation spots in California',
 'what are some popular open source projects']

In [33]:
def predict_coarse(query, lonQ_shortD_df):
    query_serialized_vec = serialize_f32(model.encode(query))
    
    retrived_results = db.execute(f"""
    with coarse_matches as (
      select
        rowid,
        embedding
      from vec_items_{persona}_2
      where embedding_coarse match vec_quantize_binary(:query_serialized_vec)
      and k=200
      order by distance
      
    )
    select
      rowid,
      vec_distance_cosine(embedding, :query_serialized_vec)
    from coarse_matches
    order by 2
    limit 2;
    """, {"query_serialized_vec": query_serialized_vec}).fetchall()
    print(retrived_results)
    return lonQ_shortD_df.iloc[[row for row,dist in retrived_results]]

In [34]:
query = longQ[0]
print(query)
predict_coarse(query, lonQ_shortD_df)

alternative medicine to manage ringing in my ear
[(1, 0.33442482352256775), (0, 0.33442482352256775)]


,doc_id,url,title,description,frecency,last_visit_date,profile,profile_id,variant
97649,a788f65f05c5ee76,http://www.rightdiagnosis.com/sym/ringing_in_e...,Ringing in ears,,281700,1736189753947959,longQ_shortD,167eb560a6601afc,length-variants
97648,06647eed20905bfb,http://symptomchecker.webmd.com/single-symptom...,Ringing in ears,,1421560,1736196807185413,longQ_shortD,167eb560a6601afc,length-variants
